# Groww TOTP Connection Test

**Auth flow:**
```
GROWW_API_TOKEN (api_key)  +  TOTP_TOKEN_SECRET (generates 6-digit OTP)
         |
         v
GrowwAPI.get_access_token(api_key, totp)  -->  daily session token
         |
         v
GrowwAPI(access_token)  -->  holdings, orders, margin, etc.
```
Access tokens expire daily at 6 AM — re-run cells 3+4 each morning.

In [ ]:
# Cell 1 — Confirm packages are installed in the correct venv
import sys
print(f'Python: {sys.executable}')

import importlib.metadata
print(f'growwapi : {importlib.metadata.version("growwapi")}')
print(f'pyotp    : {importlib.metadata.version("pyotp")}')
print()
print('If Python path above does not contain your project venv, switch')
print('the Jupyter kernel to:  venv/Scripts/python.exe')

In [ ]:
# Cell 2 — Load env vars
import os, pyotp, time
from pathlib import Path
from dotenv import load_dotenv
from growwapi import GrowwAPI

load_dotenv(Path('..') / '.env')

GROWW_API_TOKEN = os.getenv('GROWW_API_TOKEN', '')
TOTP_SECRET     = os.getenv('TOTP_TOKEN_SECRET', '')

print('GROWW_API_TOKEN   :', 'SET' if GROWW_API_TOKEN else 'MISSING')
print('TOTP_TOKEN_SECRET :', 'SET' if TOTP_SECRET     else 'MISSING')

if not GROWW_API_TOKEN: raise ValueError('GROWW_API_TOKEN missing in .env')
if not TOTP_SECRET:     raise ValueError('TOTP_TOKEN_SECRET missing in .env')

In [ ]:
# Cell 3 — Generate TOTP + get session access token
otp  = pyotp.TOTP(TOTP_SECRET).now()
secs = 30 - (int(time.time()) % 30)
print(f'OTP: {otp}  ({secs}s remaining)')

if secs < 3:
    print('OTP about to expire — waiting for next window...')
    time.sleep(secs + 1)
    otp = pyotp.TOTP(TOTP_SECRET).now()
    print(f'New OTP: {otp}')

access_token = GrowwAPI.get_access_token(api_key=GROWW_API_TOKEN, totp=otp)
client       = GrowwAPI(access_token)
print('Access token obtained — client ready')

In [ ]:
# Cell 4 — Test all API endpoints
import pprint

tests = [
    ('User profile',  lambda: client.get_user_profile()),
    ('Holdings',      lambda: client.get_holdings_for_user()),
    ('Positions',     lambda: client.get_positions_for_user()),
    ('Orders',        lambda: client.get_order_list()),
    ('Margin/Funds',  lambda: client.get_available_margin_details()),
]

for label, fn in tests:
    try:
        result = fn()
        print(f'[OK] {label}')
        pprint.pprint(result, indent=4)
    except Exception as e:
        print(f'[FAIL] {label} — {e}')
    print()

In [ ]:
# Cell 5 — Save access token to .env for reuse today (expires at 6 AM)
import re

env_path = Path('..') / '.env'
content  = env_path.read_text()

token_line = f"GROWW_ACCESS_TOKEN='{access_token}'"
if 'GROWW_ACCESS_TOKEN=' in content:
    content = re.sub(r"GROWW_ACCESS_TOKEN='[^']*'", token_line, content)
else:
    content = content.rstrip() + f'\n{token_line}\n'

env_path.write_text(content)
print('GROWW_ACCESS_TOKEN saved to .env')
print('Use os.getenv("GROWW_ACCESS_TOKEN") in any script today.')

In [ ]:
# Cell 6 — Live TOTP watch (interrupt kernel to stop)
print('Watching TOTP codes — interrupt kernel to stop')
print('-' * 40)
last = None
while True:
    code = pyotp.TOTP(TOTP_SECRET).now()
    secs = 30 - (int(time.time()) % 30)
    if code != last:
        print(f'  {code}  (valid {secs}s)')
        last = code
    time.sleep(1)